In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================

import os
import json
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf

from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

from tensorflow.keras.layers import (
    Dense,
    Dropout,
    GlobalAveragePooling2D
)

from tensorflow.keras.models import Model

from tensorflow.keras.preprocessing.image import (
    ImageDataGenerator,
    image
)

from tensorflow.keras.optimizers import Adam

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

print("TensorFlow version:", tf.__version__)

In [ ]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

IMG_SIZE = (160, 160)

IMG_ROWS = IMG_SIZE[0]
IMG_COLS = IMG_SIZE[1]

BATCH_SIZE = 16

NUM_CLASSES = 36

INITIAL_EPOCHS = 50
FINE_TUNE_EPOCHS = 20

TRAIN_DATA_DIR = "../data/fruits_data/train"
VALIDATION_DATA_DIR = "../data/fruits_data/validation"

MODEL_DIR = "../ML_models"

BEST_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "fruits_veg_identify_gpt_model.keras"
)

FINAL_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "fruits_veg_identify_v3.keras"
)

CLASS_NAMES_PATH = os.path.join(
    MODEL_DIR,
    "class_names.json"
)

os.makedirs(MODEL_DIR, exist_ok=True)

print("Image size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Number of classes:", NUM_CLASSES)
print("Training directory:", TRAIN_DATA_DIR)
print("Validation directory:", VALIDATION_DATA_DIR)

In [ ]:
# ============================================================
# 3. LOAD VGG16
# ============================================================

base_model = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_ROWS, IMG_COLS, 3)
)

print("VGG16 loaded successfully.")
print("Number of layers:", len(base_model.layers))

In [ ]:
# ============================================================
# 4. FREEZE VGG16
# ============================================================

for layer in base_model.layers:
    layer.trainable = False

print()
print("VGG16 layer status")
print("=" * 60)

for i, layer in enumerate(base_model.layers):
    print(
        f"{i:2d} "
        f"{layer.name:25s} "
        f"{layer.__class__.__name__:25s} "
        f"trainable={layer.trainable}"
    )

In [ ]:
# ============================================================
# 5. CREATE CLASSIFICATION HEAD
# ============================================================

def addTopModel(bottom_model, num_classes, D=256):

    top_model = bottom_model.output

    # Instead of Flatten()
    top_model = GlobalAveragePooling2D(
        name="global_average_pooling"
    )(top_model)

    top_model = Dense(
        D,
        activation="relu",
        name="dense_256"
    )(top_model)

    top_model = Dropout(
        0.3,
        name="dropout"
    )(top_model)

    top_model = Dense(
        num_classes,
        activation="softmax",
        name="predictions"
    )(top_model)

    return top_model

In [ ]:
# ============================================================
# 6. CREATE COMPLETE MODEL
# ============================================================

FC_Head = addTopModel(
    base_model,
    NUM_CLASSES
)

modelnew = Model(
    inputs=base_model.input,
    outputs=FC_Head,
    name="FruitVegetableVGG16"
)

modelnew.summary()

In [ ]:
# ============================================================
# 7. DATA GENERATORS
# ============================================================

train_datagen = ImageDataGenerator(

    preprocessing_function=preprocess_input,

    rotation_range=20,

    width_shift_range=0.2,

    height_shift_range=0.2,

    zoom_range=0.15,

    horizontal_flip=True,

    fill_mode="nearest"
)


validation_datagen = ImageDataGenerator(

    preprocessing_function=preprocess_input
)

In [ ]:
# ============================================================
# 8. CREATE TRAINING GENERATOR
# ============================================================

train_generator = train_datagen.flow_from_directory(

    TRAIN_DATA_DIR,

    target_size=IMG_SIZE,

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=True
)

In [ ]:
# ============================================================
# 9. CREATE VALIDATION GENERATOR
# ============================================================

validation_generator = validation_datagen.flow_from_directory(

    VALIDATION_DATA_DIR,

    target_size=IMG_SIZE,

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=False
)

In [ ]:
# ============================================================
# 10. DATASET INFORMATION
# ============================================================

num_train_classes = len(
    train_generator.class_indices
)

num_validation_classes = len(
    validation_generator.class_indices
)

print()
print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print(
    "Training images:",
    train_generator.samples
)

print(
    "Validation images:",
    validation_generator.samples
)

print(
    "Training classes:",
    num_train_classes
)

print(
    "Validation classes:",
    num_validation_classes
)

print(
    "Training batches:",
    len(train_generator)
)

print(
    "Validation batches:",
    len(validation_generator)
)

In [ ]:
# ============================================================
# 11. VERIFY CLASS MAPPINGS
# ============================================================

train_class_indices = train_generator.class_indices

validation_class_indices = (
    validation_generator.class_indices
)

print()
print("Training class mapping:")
print(train_class_indices)

print()
print("Validation class mapping:")
print(validation_class_indices)

if train_class_indices != validation_class_indices:

    raise ValueError(
        "ERROR: Training and validation class mappings are different!"
    )

print()
print("Class mappings are identical.")

In [ ]:
# ============================================================
# 12. CREATE CLASS NAMES
# ============================================================

class_names = [
    None
] * len(train_class_indices)

for class_name, class_index in train_class_indices.items():

    class_names[class_index] = class_name

print()
print("Class names:")
for index, class_name in enumerate(class_names):

    print(
        f"{index:2d}: {class_name}"
    )

print()
print("Number of classes:", len(class_names))

In [ ]:
# ============================================================
# 13. SAVE CLASS NAMES
# ============================================================

with open(
    CLASS_NAMES_PATH,
    "w"
) as file:

    json.dump(
        class_names,
        file,
        indent=4
    )

print()
print(
    "Class names saved to:",
    CLASS_NAMES_PATH
)

In [ ]:
# ============================================================
# 14. VERIFY NUMBER OF CLASSES
# ============================================================

if len(class_names) != NUM_CLASSES:

    raise ValueError(
        f"Expected {NUM_CLASSES} classes, "
        f"but found {len(class_names)} classes."
    )

print(
    f"Verified {NUM_CLASSES} classes."
)

In [ ]:
# ============================================================
# 15. COMPILE MODEL - STAGE 1
# ============================================================

modelnew.compile(

    loss="categorical_crossentropy",

    optimizer=Adam(
        learning_rate=1e-3
    ),

    metrics=[
        "accuracy"
    ]
)

print("Model compiled for Stage 1.")

In [ ]:
# ============================================================
# 16. CALLBACKS
# ============================================================

callbacks = [

    EarlyStopping(

        monitor="val_loss",

        patience=5,

        restore_best_weights=True,

        verbose=1
    ),

    ReduceLROnPlateau(

        monitor="val_loss",

        factor=0.2,

        patience=2,

        min_lr=1e-7,

        verbose=1
    ),

    ModelCheckpoint(

        BEST_MODEL_PATH,

        monitor="val_loss",

        save_best_only=True,

        verbose=1
    )
]

In [ ]:
# ============================================================
# 17. STAGE 1 TRAINING
# ============================================================

history_stage1 = modelnew.fit(

    train_generator,

    validation_data=validation_generator,

    epochs=INITIAL_EPOCHS,

    callbacks=callbacks
)

In [ ]:
# ============================================================
# 18. PLOT TRAINING HISTORY
# ============================================================

def plot_history(history, title="Training History"):

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)

    plt.plot(
        history.history["accuracy"],
        label="Training Accuracy"
    )

    plt.plot(
        history.history["val_accuracy"],
        label="Validation Accuracy"
    )

    plt.title(
        title + " - Accuracy"
    )

    plt.xlabel("Epoch")

    plt.ylabel("Accuracy")

    plt.legend()

    plt.grid(True)


    plt.subplot(1, 2, 2)

    plt.plot(
        history.history["loss"],
        label="Training Loss"
    )

    plt.plot(
        history.history["val_loss"],
        label="Validation Loss"
    )

    plt.title(
        title + " - Loss"
    )

    plt.xlabel("Epoch")

    plt.ylabel("Loss")

    plt.legend()

    plt.grid(True)

    plt.tight_layout()

    plt.show()


plot_history(
    history_stage1,
    "Stage 1"
)

In [ ]:
# ============================================================
# 19. FINE-TUNE VGG16 BLOCK 5
# ============================================================

for layer in base_model.layers:

    layer.trainable = False


for layer in base_model.layers:

    if layer.name.startswith("block5"):

        layer.trainable = True

In [ ]:
# ============================================================
# 20. CHECK FINE-TUNING LAYERS
# ============================================================

print()
print("=" * 60)
print("FINE-TUNING CONFIGURATION")
print("=" * 60)

for i, layer in enumerate(base_model.layers):

    print(
        f"{i:2d} "
        f"{layer.name:25s} "
        f"trainable={layer.trainable}"
    )

In [ ]:
# ============================================================
# 21. COMPILE MODEL - STAGE 2
# ============================================================

modelnew.compile(

    loss="categorical_crossentropy",

    optimizer=Adam(
        learning_rate=1e-5
    ),

    metrics=[
        "accuracy"
    ]
)

print(
    "Model recompiled for fine-tuning."
)

In [ ]:
# ============================================================
# 22. FINE-TUNING CALLBACKS
# ============================================================

fine_tune_callbacks = [

    EarlyStopping(

        monitor="val_loss",

        patience=5,

        restore_best_weights=True,

        verbose=1
    ),

    ReduceLROnPlateau(

        monitor="val_loss",

        factor=0.2,

        patience=2,

        min_lr=1e-7,

        verbose=1
    ),

    ModelCheckpoint(

        BEST_MODEL_PATH,

        monitor="val_loss",

        save_best_only=True,

        verbose=1
    )
]

In [ ]:
# ============================================================
# 23. STAGE 2 TRAINING
# ============================================================

history_stage2 = modelnew.fit(

    train_generator,

    validation_data=validation_generator,

    epochs=FINE_TUNE_EPOCHS,

    callbacks=fine_tune_callbacks
)

In [ ]:
# ============================================================
# 24. PLOT FINE-TUNING HISTORY
# ============================================================

plot_history(
    history_stage2,
    "Stage 2 Fine-Tuning"
)

In [ ]:
# ============================================================
# 25. SAVE FINAL MODEL
# ============================================================

modelnew.save(
    FINAL_MODEL_PATH
)

print()
print(
    "Final model saved to:"
)

print(
    FINAL_MODEL_PATH
)

In [ ]:
# ============================================================
# 26. FINAL VALIDATION EVALUATION
# ============================================================

validation_loss, validation_accuracy = (
    modelnew.evaluate(
        validation_generator,
        verbose=1
    )
)

print()
print("=" * 60)
print("FINAL VALIDATION RESULT")
print("=" * 60)

print(
    f"Validation Loss: {validation_loss:.4f}"
)

print(
    f"Validation Accuracy: "
    f"{validation_accuracy * 100:.2f}%"
)

In [ ]:
# ============================================================
# 27. SINGLE IMAGE PREDICTION
# ============================================================

def predict_single_image(img_path, top_n=10):

    print()
    print("=" * 60)
    print("EVALUATION REQUEST")
    print("=" * 60)

    print(
        "Image:",
        img_path
    )

    if not os.path.exists(img_path):

        print(
            f"ERROR: File does not exist: {img_path}"
        )

        return


    # --------------------------------------------------------
    # Load image
    # --------------------------------------------------------

    img = image.load_img(

        img_path,

        target_size=IMG_SIZE
    )


    # --------------------------------------------------------
    # Display image
    # --------------------------------------------------------

    plt.figure(
        figsize=(5, 5)
    )

    plt.imshow(img)

    plt.axis("off")

    plt.show()


    # --------------------------------------------------------
    # Convert image to NumPy array
    # --------------------------------------------------------

    img_array = image.img_to_array(img)


    # --------------------------------------------------------
    # IMPORTANT:
    # Use exactly the same preprocessing as training
    # --------------------------------------------------------

    img_array = preprocess_input(
        img_array
    )


    # --------------------------------------------------------
    # Add batch dimension
    # --------------------------------------------------------

    img_batch = np.expand_dims(
        img_array,
        axis=0
    )


    # --------------------------------------------------------
    # Predict
    # --------------------------------------------------------

    predictions = modelnew.predict(

        img_batch,

        verbose=0
    )[0]


    # --------------------------------------------------------
    # Best prediction
    # --------------------------------------------------------

    best_match_idx = np.argmax(
        predictions
    )

    confidence_score = float(
        predictions[best_match_idx]
    )

    predicted_class = (
        class_names[best_match_idx]
    )


    # --------------------------------------------------------
    # Print primary prediction
    # --------------------------------------------------------

    print()
    print("=" * 60)
    print("PREDICTION RESULT")
    print("=" * 60)

    print(
        f"Identified as: '{predicted_class}'"
    )

    print(
        f"Confidence: "
        f"{confidence_score * 100:.2f}%"
    )


    # --------------------------------------------------------
    # Top predictions
    # --------------------------------------------------------

    top_n = min(
        top_n,
        len(predictions)
    )

    top_indices = np.argsort(
        predictions
    )[-top_n:][::-1]


    print()
    print("=" * 60)
    print("TOP PREDICTIONS")
    print("=" * 60)

    for rank, idx in enumerate(
        top_indices,
        start=1
    ):

        print(
            f"{rank:2d}. "
            f"{class_names[idx]:30s} "
            f"{predictions[idx] * 100:.2f}%"
        )


    return {
        "productName": predicted_class,
        "confidence": confidence_score * 100,
        "topPredictions": [
            {
                "productName": class_names[idx],
                "confidence": float(
                    predictions[idx] * 100
                )
            }

            for idx in top_indices
        ]
    }

In [ ]:
# ============================================================
# 28. TEST IMAGE
# ============================================================

sample_test_path = "./banana1.jpg"

result = predict_single_image(
    sample_test_path,
    top_n=10
)

print()
print("Returned result:")
print(result)